# ICE-D Normalized Heatmap with Gap Bin Annotation

Visualizes ICE (Iterative Correction and Eigenvector decomposition) normalized Hi-C contact matrices with gap bin positions shown in a dedicated gap-track panel on the right side of the heatmap.

**Gap bins** are genomic regions with no Hi-C signal (assembly gaps, unmappable regions, N-character stretches). They appear as NaN/white in ICE-normalized matrices.

This notebook:
1. Loads an ICE-normalized window matrix from `.cool` files
2. Identifies gap bins using the gaps TSV annotation
3. Plots the heatmap with a **gap-track panel** (right of heatmap, left of colorbar)

---

In [ ]:
# Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cooler

# Working directory
WORKDIR = "/share/home/xhhuang24/project/PanTE/background_build"
OUTDIR  = "results"
RESOLUTION = 4000   # bin size in bp

os.chdir(WORKDIR)
print(f"Working directory: {WORKDIR}")
print(f"Results directory: {os.path.join(WORKDIR, OUTDIR)}")

---

In [ ]:
# ================================================================
# Configuration — select sample, chromosome, and window
# ================================================================
# METHOD 1: Select by window ID
SAMPLE    = "L128_rep1"
CHROM     = "HC04_A01"
WINDOW_ID = 11       # 0-based window index

# METHOD 2: Select by genomic interval
# SAMPLE    = "L128_rep1"
# CHROM     = "HC04_A01"
# WIN_START = 1000000   # bp
# WIN_END   = 1100000   # bp
# WINDOW_ID = None
# ================================================================

# --- load QC table ---
sample_chrom = f"{SAMPLE}__{CHROM}"
qc_file = os.path.join(WORKDIR, OUTDIR, "windows", f"{sample_chrom}_window_qc.tsv")
qc_df = pd.read_csv(qc_file, sep="\t", index_col=0)
print(f"[{SAMPLE} | {CHROM}] {len(qc_df)} windows loaded")

# --- resolve window ---
if WINDOW_ID is not None:
    if WINDOW_ID >= len(qc_df):
        raise ValueError(f"WINDOW_ID {WINDOW_ID} out of range (max: {len(qc_df)-1})")
    win_info = qc_df.iloc[WINDOW_ID]
    WINDOW_ID = int(WINDOW_ID)
else:
    if WIN_START is None or WIN_END is None:
        raise ValueError("WIN_START and WIN_END required in interval mode")
    matches = qc_df[(qc_df["window_start"] <= WIN_START) & (qc_df["window_end"] >= WIN_END)]
    if len(matches) == 0:
        raise ValueError(f"No window covers {WIN_START:,}-{WIN_END:,} bp")
    WINDOW_ID = int(matches.index[0])
    win_info  = matches.iloc[0]

gen_start = int(win_info["window_start"])
gen_end   = int(win_info["window_end"])
bs        = int(win_info["bin_start"])
be        = int(win_info["bin_end"])
n_gaps    = int(win_info["n_gaps"])
n_bins_qc = int(win_info["bin_count"])

print(f"\nSelected window {WINDOW_ID}: {gen_start:,}-{gen_end:,} bp")
print(f"  Local bins: {n_bins_qc}  |  Gap bins: {n_gaps}")
print(f"  Genomic bin range: {bs}–{be}  (offset={bs})")

---

In [ ]:
# Load gap bin information for this window

gaps_file = os.path.join(WORKDIR, OUTDIR, "windows", f"{sample_chrom}_gaps.tsv")
all_gaps  = pd.read_csv(gaps_file, sep="\t")

# Filter to gap bins that physically overlap the window's genomic range [gen_start, gen_end).
# Key fix: use interval-overlap semantics (gap ends after window starts AND starts before window ends)
# instead of the old buggy condition (bin_end <= gen_end), which dropped cross-boundary gaps.
win_gaps = all_gaps[
    (all_gaps["bin_end"]   > gen_start) &   # gap ends after window starts
    (all_gaps["bin_start"] < gen_end)       # gap starts before window ends
].copy()

# Convert genomic bp-start to local (window-relative) bin index
# gen_start is the genomic bp of local bin 0; resolution is bin size in bp
win_gaps["local_bin_idx"] = (win_gaps["bin_start"] - gen_start) // RESOLUTION

# Keep only gap bins that fall within the local window bin range [0, n_bins_qc)
win_gaps = win_gaps[(win_gaps["local_bin_idx"] >= 0) & (win_gaps["local_bin_idx"] < n_bins_qc)]

# Deduplicate — same gap bin may appear multiple times across adjacent windows
gap_bin_set = sorted(set(win_gaps["local_bin_idx"].tolist()))
gap_bin_arr = np.array(gap_bin_set)

print(f"Gap bins identified by overlap: {len(gap_bin_set)}")
print(f"  (QC reports n_gaps={n_gaps}; may differ due to cross-window gap assignment in gaps.tsv)")
if len(gap_bin_set) > 0:
    print(f"  Local indices: {gap_bin_set}")
else:
    print("  (none in this window)")

---

In [ ]:
# Load ICE-normalized matrix

iced_file = os.path.join(WORKDIR, OUTDIR, "balanced", f"{sample_chrom}_iced.cool")
c = cooler.Cooler(iced_file)

# Extract window region — cooler uses genomic bin indices
mat = c.matrix(balance=True, sparse=False)[bs:be, bs:be]

# Apply symmetric lower<-upper (cooler gives upper-triangular)
mat = np.maximum(mat, mat.T)

n_bins_actual = mat.shape[0]
# QC n_bins may differ from cooler matrix size — prefer actual size for plotting
if n_bins_actual != n_bins_qc:
    print(f"NOTE: QC n_bins={n_bins_qc} but cooler matrix size={n_bins_actual}. Using {n_bins_actual}.")
    n_bins = n_bins_actual
else:
    n_bins = n_bins_qc

print(f"ICE matrix shape: {mat.shape}")
print(f"  dtype:    {mat.dtype}")
print(f"  NaN count: {np.isnan(mat).sum()}  ({np.isnan(mat).mean()*100:.1f}%)")
valid = mat[~np.isnan(mat)]
if len(valid) > 0:
    print(f"  value range: [{valid.min():.4f}, {valid.max():.4f}]")
    print(f"  median: {np.median(valid):.4f}")

---

In [ ]:
# Plot: ICE heatmap + gap-track panel (gap bins on the RIGHT, not overlaid on heatmap)

def plot_iced_heatmap_with_gaps(
    mat, gap_bins, n_bins, win_id, gen_start, gen_end, resolution, n_gaps,
    sample, chrom, gap_color="#2166ac", vmin=None, vmax=None,
    cmap="YlOrRd", figsize=(11, 7), save_path=None,
):
    """
    ICE-normalized heatmap (left) + gap-track panel (right, adjacent to colorbar).
    Gap-track shows one horizontal marker per gap bin row, aligned to heatmap rows.
    """
    # auto vmin/vmax
    if vmin is None or vmax is None:
        valid = mat[~np.isnan(mat)]
        if len(valid) > 0:
            p02, p98 = np.percentile(valid, [2, 98])
            vmin = p02 if vmin is None else vmin
            vmax = p98 if vmax is None else vmax
        else:
            vmin, vmax = 0, 1

    # --- 4-column layout: [heatmap | spacer | gap-track | colorbar] ---
    gs = GridSpec(1, 4, width_ratios=[1, 0.03, 0.08, 0.04],
                  wspace=0.05, left=0.10, right=0.90, top=0.90, bottom=0.14)

    fig = plt.figure(figsize=figsize)
    ax_heat = fig.add_subplot(gs[0])
    ax_gap  = fig.add_subplot(gs[2])   # gap-track: narrow vertical panel
    ax_cbar = fig.add_subplot(gs[3])   # colorbar: far right

    # ---- main heatmap ----
    im = ax_heat.imshow(mat.astype(np.float64), cmap=cmap, aspect="equal",
                        vmin=vmin, vmax=vmax, interpolation="nearest")
    ax_heat.set_xlim(-0.5, n_bins - 0.5)
    ax_heat.set_ylim(-0.5, n_bins - 0.5)
    ax_heat.invert_yaxis()

    # tick labels
    tick_step = max(1, n_bins // 10)
    tick_pos  = list(range(0, n_bins, tick_step))
    tick_labels = [f"{(gen_start + i * resolution) / 1e6:.1f}Mb" for i in tick_pos]
    ax_heat.set_xticks(tick_pos)
    ax_heat.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)
    ax_heat.set_yticks(tick_pos)
    ax_heat.set_yticklabels(tick_labels, fontsize=8)
    ax_heat.set_xlabel("Genomic position", fontsize=9)
    ax_heat.set_ylabel("Genomic position", fontsize=9)
    ax_heat.set_title(
        f"ICE Normalized — {sample} | {chrom} | Window {win_id}\n"
        f"{gen_start:,}-{gen_end:,} bp  ({n_bins} bins, {len(gap_bins)} gap bins)",
        fontsize=10, pad=8)

    # ---- gap-track panel (right side, between heatmap and colorbar) ----
    gap_set = set(gap_bins)
    ax_gap.set_xlim(0, 1)
    ax_gap.set_ylim(-0.5, n_bins - 0.5)
    ax_gap.invert_yaxis()
    ax_gap.set_xticks([])
    ax_gap.set_yticks([])
    ax_gap.set_facecolor("#f7f7f7")

    for row in range(n_bins):
        color = gap_color if row in gap_set else "#e0e0e0"
        ax_gap.axhspan(row - 0.5, row + 0.5, xmin=0, xmax=1, color=color, linewidth=0)

    ax_gap.text(0.5, -1.2, "Gap", transform=ax_gap.transAxes,
                ha="center", va="top", fontsize=8, color="#555555")
    for side in ["top", "bottom", "left", "right"]:
        ax_gap.spines[side].set_color("#cccccc")
        ax_gap.spines[side].set_linewidth(0.8)

    # ---- colorbar ----
    cbar = plt.colorbar(im, cax=ax_cbar)
    cbar.set_label("ICE contact strength", fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    ax_gap.spines["right"].set_visible(False)
    ax_cbar.spines["left"].set_color("#cccccc")

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()
    return fig


# draw
fig = plot_iced_heatmap_with_gaps(
    mat=mat, gap_bins=gap_bin_arr, n_bins=n_bins, win_id=WINDOW_ID,
    gen_start=gen_start, gen_end=gen_end, resolution=RESOLUTION,
    n_gaps=n_gaps, sample=SAMPLE, chrom=CHROM,
    gap_color="#2166ac", cmap="YlOrRd", figsize=(11, 7),
)